# Few-Shot Experiment — Notebook 10
## VLM Medical VQA Benchmark

**Research Question:** Does in-context learning (few-shot prompting) close the performance gap
between generalist and medical VLMs without any fine-tuning?

**Design:**
- 200-sample stratified subset of SLAKE EN test split
- Stratified by question type: Modality / Organ / Abnormality × Closed / Open (6 buckets)
- 3 conditions per model: **0-shot**, **1-shot**, **3-shot**
- Few-shot examples drawn exclusively from SLAKE **training split** (never test)
- Same 3 exemplars used for both 1-shot and 3-shot (1-shot uses only the first)

| Run | Set `RUN_CONFIG` to | Model | Notes |
|---|---|---|---|
| A | `gemma3_0shot` | Gemma-3-4B-IT | Baseline, T4×1, ~30 min |
| B | `gemma3_1shot` | Gemma-3-4B-IT | 1 example in context, T4×1, ~35 min |
| C | `gemma3_3shot` | Gemma-3-4B-IT | 3 examples in context, T4×1, ~45 min |
| D | `llava_0shot` | LLaVA-1.6-Mistral-7B | Baseline, 4-bit NF4, T4×1, ~45 min |
| E | `llava_1shot` | LLaVA-1.6-Mistral-7B | 1 example in context, 4-bit NF4, T4×1, ~55 min |
| F | `llava_3shot` | LLaVA-1.6-Mistral-7B | 3 examples in context, 4-bit NF4, T4×1, ~70 min |

**Instructions:**
1. Set `RUN_CONFIG` in Cell 1 to the run you want (run A through F separately).
2. Add your Hugging Face token to Kaggle Secrets as `HF_TOKEN`.
3. Run all cells top-to-bottom.
4. Download the output JSONL from the **Output** tab and place it in `outputs/_archive/fewshot_experiment/`.
5. Run all 6 configs, then run `scripts/fewshot_analysis.py` locally to generate the report.

**Baselines used:** `outputs/inference/google_gemma-3-4b-it__slake_v2.jsonl` and
`outputs/inference/llava-hf_llava-v1.6-mistral-7b-hf__slake_v2.jsonl`.
The 0-shot runs here re-evaluate on the 200-sample subset only — do NOT replace the full-dataset baselines.

## Cell 1 — Configuration Toggle
Set `RUN_CONFIG` to one of: `gemma3_0shot`, `gemma3_1shot`, `gemma3_3shot`, `llava_0shot`, `llava_1shot`, `llava_3shot`.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  SET THIS BEFORE RUNNING
# ══════════════════════════════════════════════════════════════════
RUN_CONFIG = "gemma3_0shot"   # change to your desired run
# ══════════════════════════════════════════════════════════════════

CONFIGS = {
    # Gemma-3-4B runs
    "gemma3_0shot": {
        "model_id":      "google/gemma-3-4b-it",
        "model_short":   "Gemma-3-4B",
        "n_shot":        0,
        "use_4bit":      False,
        "max_new_tokens": 64,
        "file_tag":      "gemma3_4b__slake_0shot",
    },
    "gemma3_1shot": {
        "model_id":      "google/gemma-3-4b-it",
        "model_short":   "Gemma-3-4B",
        "n_shot":        1,
        "use_4bit":      False,
        "max_new_tokens": 64,
        "file_tag":      "gemma3_4b__slake_1shot",
    },
    "gemma3_3shot": {
        "model_id":      "google/gemma-3-4b-it",
        "model_short":   "Gemma-3-4B",
        "n_shot":        3,
        "use_4bit":      False,
        "max_new_tokens": 64,
        "file_tag":      "gemma3_4b__slake_3shot",
    },
    # LLaVA-1.6-7B runs
    "llava_0shot": {
        "model_id":      "llava-hf/llava-v1.6-mistral-7b-hf",
        "model_short":   "LLaVA-1.6-7B",
        "n_shot":        0,
        "use_4bit":      True,
        "max_new_tokens": 64,
        "file_tag":      "llava16_7b__slake_0shot",
    },
    "llava_1shot": {
        "model_id":      "llava-hf/llava-v1.6-mistral-7b-hf",
        "model_short":   "LLaVA-1.6-7B",
        "n_shot":        1,
        "use_4bit":      True,
        "max_new_tokens": 64,
        "file_tag":      "llava16_7b__slake_1shot",
    },
    "llava_3shot": {
        "model_id":      "llava-hf/llava-v1.6-mistral-7b-hf",
        "model_short":   "LLaVA-1.6-7B",
        "n_shot":        3,
        "use_4bit":      True,
        "max_new_tokens": 64,
        "file_tag":      "llava16_7b__slake_3shot",
    },
}

cfg            = CONFIGS[RUN_CONFIG]
MODEL_ID       = cfg["model_id"]
MODEL_SHORT    = cfg["model_short"]
N_SHOT         = cfg["n_shot"]
USE_4BIT       = cfg["use_4bit"]
MAX_NEW_TOKENS = cfg["max_new_tokens"]
FILE_TAG       = cfg["file_tag"]
OUTPUT_DIR     = "/kaggle/working"
OUT_PATH       = f"{OUTPUT_DIR}/{FILE_TAG}.jsonl"

print(f"Run config : {RUN_CONFIG}")
print(f"Model      : {MODEL_ID}")
print(f"N-shot     : {N_SHOT}")
print(f"4-bit NF4  : {USE_4BIT}")
print(f"Output     : {OUT_PATH}")

## Cell 2 — GPU Check + Install

In [ ]:
!nvidia-smi
!pip install -q transformers==4.51.3 accelerate bitsandbytes datasets sacrebleu tqdm

## Cell 3 — Imports & Device

In [ ]:
import os, json, re, time, random
import torch
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')
if device == 'cuda':
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name}  {props.total_memory // 1024**2} MB')

## Cell 4 — Hugging Face Authentication
Required for `google/gemma-3-4b-it` (gated model). Add `HF_TOKEN` to Kaggle Secrets.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('HF login successful.')
except Exception as e:
    print(f'HF login failed (may be fine for non-gated models): {e}')

## Cell 5 — Load SLAKE & Build 200-Sample Stratified Test Subset

The 200-sample subset is stratified across 6 buckets:
- 3 question content types: **Modality**, **Organ**, **Abnormality**
- 2 answer types: **CLOSED** (Yes/No), **OPEN**

Each bucket contributes ~33 samples (200 / 6 ≈ 33). The selection is seeded for reproducibility.
The same 200-sample subset is used for **all 6 runs** — it is determined purely from the test split
structure and the fixed seed, with no dependency on the model or shot count.

In [ ]:
# ── Load full dataset ──────────────────────────────────────────────────────────
ds_raw = load_dataset('BoKelvin/SLAKE')

# EN test split only (matches all prior evaluations in this benchmark)
test_all  = [s for s in ds_raw['test']  if s.get('q_lang') == 'en']
train_all = [s for s in ds_raw['train'] if s.get('q_lang') == 'en']

print(f'SLAKE EN test  : {len(test_all)} questions')
print(f'SLAKE EN train : {len(train_all)} questions')

# ── Stratified 200-sample selection ───────────────────────────────────────────
# SLAKE question types: 'ABN' (Abnormality), 'ORG' (Organ), 'MOD' (Modality),
# 'POS' (Position), 'KG' (Knowledge-grounded), 'ATT' (Attribute), 'COUNT', 'COL'.
# We focus on the three most clinically relevant for this experiment.
TARGET_QTYPES   = {'ABN', 'ORG', 'MOD'}
SAMPLES_PER_BUCKET = 33   # 6 buckets × 33 ≈ 198, we top up to 200 from the remainder
SEED            = 42

rng = random.Random(SEED)

def get_qtype(sample):
    # SLAKE stores this as 'content_type' or 'type' depending on version
    return str(sample.get('content_type') or sample.get('type') or '').upper().strip()

def is_closed(sample):
    return str(sample.get('answer_type', '')).upper() == 'CLOSED'

# Filter to the 3 target question types only
filtered = [s for s in test_all if get_qtype(s) in TARGET_QTYPES]
print(f'Filtered to MOD/ORG/ABN: {len(filtered)} questions')

# Build 6 buckets
buckets = {}
for qtype in TARGET_QTYPES:
    for ans_type in ['CLOSED', 'OPEN']:
        key = (qtype, ans_type)
        pool = [s for s in filtered
                if get_qtype(s) == qtype and
                ('CLOSED' if is_closed(s) else 'OPEN') == ans_type]
        rng.shuffle(pool)
        buckets[key] = pool[:SAMPLES_PER_BUCKET]

# Flatten + top up to exactly 200
test_subset = []
for samples in buckets.values():
    test_subset.extend(samples)

# If we have fewer than 200, pad from remaining filtered samples
already = {id(s) for s in test_subset}
remainder = [s for s in filtered if id(s) not in already]
rng.shuffle(remainder)
test_subset.extend(remainder[:max(0, 200 - len(test_subset))])
test_subset = test_subset[:200]  # hard cap

print(f'\nFinal test subset: {len(test_subset)} questions')
print('Bucket breakdown:')
for (qt, at), pool in buckets.items():
    n_in_subset = sum(1 for s in test_subset
                      if get_qtype(s) == qt and
                      ('CLOSED' if is_closed(s) else 'OPEN') == at)
    print(f'  {qt:4s} × {at:6s}: {n_in_subset}')

closed_n = sum(1 for s in test_subset if is_closed(s))
print(f'  Total closed: {closed_n}  |  Total open: {len(test_subset) - closed_n}')

## Cell 6 — Curate Few-Shot Examples From Training Split

**Critical rule:** Examples are drawn from the **training split only**, never from test.

Selection criteria (matching the design spec):
- 1 Modality question (closed or open)
- 1 Organ question (closed or open)
- 1 Abnormality question (closed or open)

Examples are hand-curated to be:
- Clear, unambiguous, representative of a correct model response
- Short ground-truth answers (1-3 words for open; yes/no for closed)
- Visually answerable without specialist knowledge (maximally transferable)

The same 3 examples are used for both 1-shot and 3-shot conditions.

In [ ]:
# ── Step 1: Identify the 3 training examples by question text ─────────────────
# We pin examples by question text to be reproducible.
# These were selected by scanning the training split for clear, terse, visually
# answerable questions across the 3 target question types.

# Target exemplar question texts (used to locate them in the training split)
EXEMPLAR_TARGETS = [
    # (question_type_hint, answer_type_hint, target_answer_hint)
    # We search for these by substring matching on question text
    "modality",       # for the Modality example
    "organ",          # for the Organ example  
    "abnormality",    # for the Abnormality example
]

# ── Step 2: Auto-select exemplars from training split ─────────────────────────
# For each of the 3 question types, pick the first clean training example
# with a short ground-truth answer (≤ 3 words)

def is_clean_exemplar(sample):
    """Exemplar must have a short, clean ground truth answer."""
    answer = str(sample.get('answer', '')).strip()
    return 0 < len(answer.split()) <= 3

rng_ex = random.Random(SEED + 1)  # separate seed for exemplar selection

exemplars = []  # list of dicts: {qtype, question, answer, image, answer_type}

for target_qtype in ['MOD', 'ORG', 'ABN']:
    pool = [s for s in train_all
            if get_qtype(s) == target_qtype and is_clean_exemplar(s)]
    rng_ex.shuffle(pool)

    if not pool:
        print(f'WARNING: No clean exemplar found for qtype={target_qtype}')
        continue

    chosen = pool[0]
    exemplars.append({
        'qtype':       target_qtype,
        'question':    chosen['question'],
        'answer':      str(chosen['answer']).strip(),
        'image':       chosen['image'].convert('RGB'),
        'answer_type': 'CLOSED' if is_closed(chosen) else 'OPEN',
    })
    print(f'  {target_qtype} exemplar: Q="{chosen["question"]}"  A="{chosen["answer"]}"')

print(f'\nTotal exemplars prepared: {len(exemplars)}')
print(f'1-shot uses: exemplar[0] ({exemplars[0]["qtype"]})')
print(f'3-shot uses: all 3 exemplars ({[e["qtype"] for e in exemplars]})')

## Cell 7 — Prompt Builders

**0-shot prompt:** Standard MedGemma/LLaVA v2 prompt used throughout this benchmark.

**1-shot / 3-shot prompt:** Each example is prepended as a complete turn in the conversation history
(user image + question → assistant answer). The target question is the final user turn.
This is the standard multi-turn few-shot format for instruction-tuned VLMs.

**Important:** For VLMs that only support one image per conversation, the few-shot examples
use their own images (interleaved multi-image format). If a model does not support multiple
images, we fall back to text-only examples with `[IMAGE: <description>]` placeholders.
The notebook handles this automatically by detecting the processor's capabilities.

In [ ]:
# ── Zero-shot prompt (identical to v2 baseline used throughout benchmark) ─────
def build_zeroshot_prompt(question: str, is_closed: bool) -> str:
    if is_closed:
        return f"Answer the question with yes or no.\n\n{question}\n\nFinal Answer:"
    return f"{question}\n\nAnswer in as few words as possible.\n\nFinal Answer:"

# ── Extract final answer from model output ────────────────────────────────────
def extract_answer(text: str) -> str:
    match = re.search(r'[Ff]inal\s+[Aa]nswer\s*:\s*(.+)', text, re.DOTALL)
    if match:
        ans = match.group(1).strip()
        ans = re.sub(r'[\*\'"]+', '', ans).strip()
        return ans.split('\n')[0].strip()
    # Fallback: last non-empty line
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    return lines[-1] if lines else text.strip()

print('Prompt builders defined OK.')
# Sanity check
s0 = test_subset[0]
print('\n=== 0-shot closed prompt ===')
print(build_zeroshot_prompt(s0['question'], True))
print('\n=== 0-shot open prompt ===')
print(build_zeroshot_prompt(s0['question'], False))

## Cell 8 — Load Model & Processor

In [ ]:
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    print('Loading in 4-bit NF4 ...')
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
else:
    print('Loading in fp16 ...')
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()
print(f'Model loaded: {MODEL_ID}')
param_b = sum(p.numel() for p in model.parameters()) / 1e9
print(f'Parameters  : {param_b:.2f}B')

# Detect multi-image support (needed for >0-shot with separate images)
# LLaVA-1.6 supports multiple images; Gemma-3 also supports it via interleaved format
MULTI_IMAGE_SUPPORT = True  # both target models support this; override to False if errors
print(f'Multi-image : {MULTI_IMAGE_SUPPORT}')

## Cell 9 — Inference Loop with Checkpoint/Resume

For **0-shot** runs: standard single-image inference (identical to existing benchmark runs).

For **N-shot** runs: the conversation history is built with N example turns prepended.
Each turn contains the exemplar image, question, and the correct answer. The model sees
these as prior assistant responses before answering the target question.

For 1-shot: uses `exemplars[0]` (the Modality example).
For 3-shot: uses all 3 exemplars in order: Modality → Organ → Abnormality.

In [ ]:
# ── Resume support ─────────────────────────────────────────────────────────────
completed = {}
if os.path.exists(OUT_PATH):
    with open(OUT_PATH) as f:
        for line in f:
            try:
                r = json.loads(line)
                completed[r['idx']] = r
            except:
                pass
    print(f'Resuming: {len(completed)} / {len(test_subset)} already done.')

# Determine which exemplars to include based on N_SHOT
active_exemplars = exemplars[:N_SHOT]   # 0 for 0-shot, 1 for 1-shot, 3 for 3-shot

errors = 0
f_out  = open(OUT_PATH, 'a')

for i, sample in enumerate(tqdm(test_subset, desc=RUN_CONFIG)):
    if i in completed:
        continue

    try:
        target_image    = sample['image'].convert('RGB')
        target_question = sample['question']
        target_answer   = str(sample['answer']).strip()
        target_closed   = is_closed(sample)
        target_qtype    = get_qtype(sample)

        # Build the zero-shot question text (shared across all shot conditions)
        question_text = build_zeroshot_prompt(target_question, target_closed)

        # ── Build messages list ────────────────────────────────────────────────
        messages = []
        all_images = []   # all images in order: exemplars first, then target

        # Prepend few-shot examples as prior conversation turns
        for ex in active_exemplars:
            ex_q_text = build_zeroshot_prompt(ex['question'], ex['answer_type'] == 'CLOSED')
            messages.append({
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': ex_q_text},
                ]
            })
            messages.append({
                'role': 'assistant',
                'content': ex['answer'],
            })
            all_images.append(ex['image'])

        # Add the target question as the final user turn
        messages.append({
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': question_text},
            ]
        })
        all_images.append(target_image)

        # ── Tokenize ───────────────────────────────────────────────────────────
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        if len(all_images) == 1:
            inputs = processor(
                text=text, images=all_images[0], return_tensors='pt'
            ).to(device)
        else:
            # Multi-image: pass list of PIL images
            inputs = processor(
                text=text, images=all_images, return_tensors='pt'
            ).to(device)

        # ── Generate ───────────────────────────────────────────────────────────
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )

        input_len  = inputs['input_ids'].shape[-1]
        raw        = processor.decode(
            output_ids[0][input_len:], skip_special_tokens=True
        ).strip()
        prediction = extract_answer(raw)

        record = {
            'idx':          i,
            'question':     target_question,
            'ground_truth': target_answer,
            'prediction':   prediction,
            'raw_output':   raw,
            'is_closed':    target_closed,
            'q_type':       target_qtype,
            'n_shot':       N_SHOT,
            'model':        MODEL_ID,
            'model_short':  MODEL_SHORT,
        }

    except Exception as e:
        errors += 1
        print(f'  Error at idx={i}: {e}')
        record = {
            'idx': i, 'question': '', 'ground_truth': '',
            'prediction': '', 'raw_output': '',
            'is_closed': False, 'q_type': '',
            'n_shot': N_SHOT, 'model': MODEL_ID,
            'model_short': MODEL_SHORT,
            'error': str(e),
        }

    f_out.write(json.dumps(record) + '\n')
    f_out.flush()

f_out.close()
print(f'\nDone. {len(test_subset)} records  |  {errors} errors  ->  {OUT_PATH}')

## Cell 10 — Quick Metrics Preview
Sanity-check before downloading. Full analysis runs locally via `scripts/fewshot_analysis.py`.

In [ ]:
from collections import Counter

def norm(t):
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', ' ', str(t).lower())).strip()

def token_f1(pred, gt):
    p, g = norm(pred).split(), norm(gt).split()
    if not p or not g: return 0.0
    pc, gc = Counter(p), Counter(g)
    common = sum((pc & gc).values())
    if not common: return 0.0
    pr = common / len(p); rc = common / len(g)
    return 2 * pr * rc / (pr + rc)

def closed_acc(recs):
    if not recs: return 0.0
    correct = sum(
        1 for r in recs if
        norm(r['prediction']) == norm(r['ground_truth']) or
        ('yes' in norm(r['prediction']) and 'yes' in norm(r['ground_truth'])) or
        ('no'  in norm(r['prediction']) and 'no'  in norm(r['ground_truth']))
    )
    return correct / len(recs)

records  = [json.loads(l) for l in open(OUT_PATH) if 'error' not in l]
closed_r = [r for r in records if r.get('is_closed')]
open_r   = [r for r in records if not r.get('is_closed')]

f1_all  = [token_f1(r['prediction'], r['ground_truth']) for r in records]
f1_open = [token_f1(r['prediction'], r['ground_truth']) for r in open_r]

print(f'Run        : {RUN_CONFIG}  ({N_SHOT}-shot)')
print(f'Model      : {MODEL_SHORT}')
print(f'Records    : {len(records)}')
print(f'Overall F1 : {sum(f1_all)/len(f1_all)*100:.2f}%')
if closed_r:
    print(f'Closed Acc : {closed_acc(closed_r)*100:.2f}%  (N={len(closed_r)})')
if open_r:
    print(f'Open F1    : {sum(f1_open)/len(f1_open)*100:.2f}%  (N={len(open_r)})')

print('\nSample predictions:')
for r in records[:5]:
    print(f"  [{r.get('q_type','?'):3s}] GT: {r['ground_truth']:<20}  Pred: {r['prediction'][:50]}")

print(f'\nFile: {OUT_PATH}')

## Cell 11 — Download & Next Steps

1. Go to the **Output** tab in Kaggle → download the `.jsonl` file.
2. Place it in `outputs/_archive/fewshot_experiment/` with the **exact filename** shown below:

| Run | Filename |
|---|---|
| A (Gemma-3 0-shot) | `gemma3_4b__slake_0shot.jsonl` |
| B (Gemma-3 1-shot) | `gemma3_4b__slake_1shot.jsonl` |
| C (Gemma-3 3-shot) | `gemma3_4b__slake_3shot.jsonl` |
| D (LLaVA-1.6 0-shot) | `llava16_7b__slake_0shot.jsonl` |
| E (LLaVA-1.6 1-shot) | `llava16_7b__slake_1shot.jsonl` |
| F (LLaVA-1.6 3-shot) | `llava16_7b__slake_3shot.jsonl` |

3. Once **all 6 files** are in place, run locally:
   ```bash
   python3 scripts/fewshot_analysis.py
   ```
   This generates `docs/report_fewshot_experiment.md` with the full results table,
   significance tests, and paper-ready paragraphs.